# Train "hey millie" on Indian voices only (Sarvam dataset)

**Runtime → T4 GPU.** Upload two files to `/content` before running:
- `sarvam_hey_millie_16k.zip`: 768 positive clips (256 Sarvam recordings × 3 slight variations), already split by speaker
- `millie_voice_samples.zip`: your own recordings, used **only for testing** (never trained on)

**What's different from the earlier run:**
- **Positive examples:** only the Sarvam Indian voices. No Piper (American) "hey millie" clips are generated.
- **Train/test split by speaker:** 60 speakers train (651 clips), 23 different speakers validate (117 clips). No voice appears on both sides, so validation recall measures new speakers, not memorised ones.
- **Negatives:** unchanged. Similar-sounding phrases from Piper TTS (only teach "this is *not* hey millie", accent doesn't matter here) + ~2,000 h of general audio.
- **Your recordings:** a fully independent real-voice test at the end.


## Expected time on a free T4 (estimates)

| Step | Approx. time |
|---|---|
| Clone + Python 3.10 environment | 4–6 min |
| Data downloads (skipped if already present) | 10–15 min |
| Negative clips via TTS (skipped if they exist from the last run) | 5–8 min |
| Augmentation + features | 4–8 min |
| Training (10,000 steps) + export | 15–25 min |


In [1]:
%cd /content
!rm -rf piper-sample-generator openwakeword
!git clone -q https://github.com/rhasspy/piper-sample-generator && cd piper-sample-generator && git checkout -q v2.0.0
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!git clone -q https://github.com/dscripka/openwakeword
!mkdir -p openwakeword/openwakeword/resources/models
!for f in embedding_model.onnx embedding_model.tflite melspectrogram.onnx melspectrogram.tflite; do wget -q -O openwakeword/openwakeword/resources/models/$f https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/$f; done
!ls -la piper-sample-generator/models openwakeword/openwakeword/resources/models

/content
openwakeword/openwakeword/resources/models:
total 4736
drwxr-xr-x 2 root root    4096 Nov 26 08:02 .
drwxr-xr-x 3 root root    4096 Nov 26 08:02 ..
-rw-r--r-- 1 root root 1326578 Oct  9  2023 embedding_model.onnx
-rw-r--r-- 1 root root 1330312 Oct  9  2023 embedding_model.tflite
-rw-r--r-- 1 root root 1087958 Oct  9  2023 melspectrogram.onnx
-rw-r--r-- 1 root root 1092516 Oct  9  2023 melspectrogram.tflite

piper-sample-generator/models:
total 199368
drwxr-xr-x 2 root root      4096 Nov 26 08:02 .
drwxr-xr-x 7 root root      4096 Nov 26 08:02 ..
-rw-r--r-- 1 root root     24484 Nov 26 08:02 en-us-libritts-high.pt.json
-rw-r--r-- 1 root root 204089915 Oct  6  2023 en_US-libritts_r-medium.pt
-rwxr-xr-x 1 root root     25782 Nov 26 08:02 en_US-libritts_r-medium.pt.json
-rw-r--r-- 1 root root         0 Nov 26 08:02 .gitkeep


## Python 3.10 environment (takes ~3–5 min)
Versions below were resolved and tested end-to-end (generate → augment → train → onnx + tflite export).

In [2]:
%%writefile /content/requirements_oww.txt
torch==2.1.2
torchaudio==2.1.2
-e ./openwakeword
piper-phonemize==1.1.0
webrtcvad
numpy<1.24
setuptools<81
pyarrow<15
mutagen==1.47.0
torchinfo==1.8.0
torchmetrics==1.2.0
speechbrain==0.5.14
audiomentations==0.33.0
torch-audiomentations==0.11.0
acoustics==0.2.6
tensorflow-cpu==2.8.1
tensorflow_probability==0.16.0
onnx_tf==1.10.0
onnx==1.14.0
protobuf>=3.20,<4
pronouncing==0.2.0
datasets==2.14.6
deep-phonemizer==0.0.19
pyyaml

Writing /content/requirements_oww.txt


In [3]:
!pip -q install uv
![ -x /content/oww-env/bin/python ] || uv venv -q --python 3.10 /content/oww-env
!uv pip install -q --python /content/oww-env/bin/python -r /content/requirements_oww.txt
PY = "/content/oww-env/bin/python"
import os
os.environ["MPLBACKEND"] = "Agg"  # Colab's inline plot backend doesn't exist in the 3.10 env
!{PY} -W ignore -c "import torch, torchaudio, tensorflow as tf; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| tf', tf.__version__)" 2>&1 | tail -1

torch 2.1.2+cu121 | CUDA: True | tf 2.8.1


## Download training data
Room impulse responses, background audio (AudioSet + FMA), and precomputed negative features (~16 GB).

In [4]:
%%writefile /content/download_data.py
"""Download room impulse responses and background audio, saved as 16 kHz 16-bit WAV."""
import io
import os
import subprocess
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import scipy.io.wavfile as wavfile
import soundfile as sf
from scipy.signal import resample_poly
from tqdm import tqdm
import datasets

SR = 16000


def save(path, x, sr):
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        g = np.gcd(int(sr), SR)
        x = resample_poly(x, SR // g, int(sr) // g)
    wavfile.write(path, SR, (np.clip(x, -1, 1) * 32767).astype(np.int16))


# 1) MIT room impulse responses
out = Path("mit_rirs"); out.mkdir(exist_ok=True)
if len(list(out.glob("*.wav"))) < 270:
    ds = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(ds, desc="RIRs"):
        save(out / row["audio"]["path"].split("/")[-1], row["audio"]["array"], row["audio"]["sampling_rate"])

# 2) AudioSet: the HF repo moved from .tar files to parquet (data/bal_train/00..37.parquet, ~700 MB each)
def get_audioset():
    out = Path("audioset_16k"); out.mkdir(exist_ok=True)
    if not any(out.glob("*.wav")):
        pfile = "audioset_bal_train_09.parquet"
        if not os.path.exists(pfile):
            subprocess.run(["wget", "-q", "-O", pfile,
                            "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet"],
                           check=True)
        pf = pq.ParquetFile(pfile)
        # find the audio column: a struct with a 'bytes' field
        audio_col = next(f.name for f in pf.schema_arrow
                         if hasattr(f.type, "num_fields") and "bytes" in [f.type.field(i).name for i in range(f.type.num_fields)])
        n = 0
        for batch in tqdm(pf.iter_batches(columns=[audio_col], batch_size=64), desc="AudioSet"):
            for item in batch.column(0).to_pylist():
                if not item or not item.get("bytes"):
                    continue
                try:
                    x, sr = sf.read(io.BytesIO(item["bytes"]), dtype="float32")
                except Exception:
                    continue
                name = Path(item.get("path") or f"clip_{n}").stem + ".wav"
                save(out / name, x, sr)
                n += 1
        print(f"AudioSet: saved {n} clips")
        os.remove(pfile)


try:
    get_audioset()
except Exception as e:
    print(f"AudioSet download failed ({e}); continuing with FMA only. Training still works.")

# 3) Free Music Archive, 1 hour of 30 s clips (increase n_hours for better models)
out = Path("fma"); out.mkdir(exist_ok=True)
n_hours = 1
if len(list(out.glob("*.wav"))) < n_hours * 120:
    ds = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    ds = iter(ds.cast_column("audio", datasets.Audio(sampling_rate=SR)))
    for _ in tqdm(range(n_hours * 3600 // 30), desc="FMA"):
        row = next(ds)
        save(out / row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav"), row["audio"]["array"], SR)

for d in ["mit_rirs", "audioset_16k", "fma"]:
    print(d, len(list(Path(d).glob("*.wav"))), "files")


Writing /content/download_data.py


In [5]:
%%time
%cd /content
!{PY} -W ignore download_data.py
![ -f openwakeword_features_ACAV100M_2000_hrs_16bit.npy ] || wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
![ -f validation_set_features.npy ] || wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
!ls -la *.npy

/content




AudioSet: saved 500 clips



mit_rirs 270 files
audioset_16k 500 files
fma 120 files
-rw-r--r-- 1 root root 17280000128 Nov 26 08:09 openwakeword_features_ACAV100M_2000_hrs_16bit.npy
-rw-r--r-- 1 root root   184836608 Nov 26 08:09 validation_set_features.npy
CPU times: user 562 ms, sys: 87.1 ms, total: 649 ms
Wall time: 5min 9s


## Load the Sarvam dataset

Clears the old positives (Piper + your-voice copies) and the old features, then unpacks the Sarvam clips. Negative clips from a previous run are kept and reused.

In [6]:
%cd /content
import os
assert os.path.exists("/content/sarvam_hey_millie_16k.zip"), "Upload sarvam_hey_millie_16k.zip to /content first"
M = "my_custom_model/hey_millie"
!rm -rf {M}/positive_train {M}/positive_test {M}/*.npy my_custom_model/hey_millie.onnx my_custom_model/hey_millie.tflite
!mkdir -p {M} && unzip -q -o /content/sarvam_hey_millie_16k.zip -d /tmp/sarvam && cp -r /tmp/sarvam/sarvam_hey_millie_16k/positive_train /tmp/sarvam/sarvam_hey_millie_16k/positive_test {M}/
n_train = len(os.listdir(f"{M}/positive_train")); n_test = len(os.listdir(f"{M}/positive_test"))
print("positive_train:", n_train, "| positive_test:", n_test)

/content
positive_train: 651 | positive_test: 117


## Training config

In [7]:
import yaml
from pathlib import Path
config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
config["target_phrase"] = ["hey millie", "hey milly", "hay millie"]  # only used to generate similar-sounding NEGATIVES
config["model_name"] = "hey_millie"
config["n_samples"] = n_train     # = number of Sarvam clips, so train.py generates NO Piper positives
config["n_samples_val"] = n_test  # = number of held-out-speaker Sarvam clips
config["steps"] = 10000         # drop to 5000 if you only want a quick first model
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["piper_sample_generator_path"] = "./piper-sample-generator"
config["output_dir"] = "./my_custom_model"
config["rir_paths"] = ["./mit_rirs"]
# only use background folders that actually got files
config["background_paths"] = [d for d in ["./audioset_16k", "./fma"] if any(Path(d).glob("*.wav"))]
config["background_paths_duplication_rate"] = [1] * len(config["background_paths"])
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
yaml.dump(config, open("my_model.yaml", "w"))
print("background:", config["background_paths"])
print("n_samples:", config["n_samples"], "| n_samples_val:", config["n_samples_val"])

background: ['./audioset_16k', './fma']
n_samples: 651 | n_samples_val: 117


## Train

In [8]:
%%time
# Step 1: should print "Skipping generation of positive clips" for both train and test.
# Negatives are reused if they exist, otherwise generated (similar-sounding phrases).
!{PY} -W ignore openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

torchvision is not available - cannot save figures
INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:CUDA available, using GPU
DEBUG:generate_samples:Batch 1/93 complete
DEBUG:generate_samples:Batch 2/93 complete
DEBUG:generate_samples:Batch 3/93 complete
DEBUG:generate_samples:Batch 4/93 complete
DEBUG:generate_samples:Batch 5/93 complete
DEBUG:generate_samples:Batch 6/93 complete
DEBUG:generate_samples:Batch 7/93 complet

In [9]:
%%time
# Step 2: --overwrite so features are rebuilt from the Sarvam clips
!{PY} -W ignore openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips --overwrite
!ls -la my_custom_model/hey_millie/*.npy

torchvision is not available - cannot save figures
INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################








-rw-r--r-- 1 root root  718976 Nov 26 08:11 my_custom_model/hey_millie/negative_features_test.npy
-rw-r--r-- 1 root root 3999872 Nov 26 08:11 my_custom_model/hey_millie/negative_features_train.npy
-rw-r--r-- 1 root root  718976 Nov 26 08:11 my_custom_model/hey_millie/positive_features_test.npy
-rw-r--r-- 1 root root 3999872 Nov 26 08:10 my_custom_model/hey_millie/positive_features_train.npy
CPU times: user 166 ms, sys: 11.1 ms, total: 177 ms
Wall time: 1min 18s


In [10]:
%%time
# Step 3: train; writes my_custom_model/hey_millie.onnx and .tflite
!{PY} -W ignore openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model
!ls -la my_custom_model/*.onnx my_custom_model/*.tflite

torchvision is not available - cannot save figures
INFO:root:##################################################
Starting training sequence 1...
##################################################

INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...

INFO:root:##################################################
Starting training sequence 3...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...

INFO:root:Merging checkpoints above the 90th percentile into single model...
INFO:root:
################
Final Model Accuracy: 0.5555555820465088
Final Model Recall: 0.1111111119389534
Final Model False Positives per Hour: 4.77876091003418
################

INFO:root:####
Saving ONNX mode as '/content/my_custom_model/hey_millie.onnx'
DEBUG:tensorflow:Falli

## Test on your own voice (never seen in training)

All of your recordings are now an independent test set: different speaker, real microphone, not synthetic. This is the number that matters for daily use.

In [11]:
%%writefile /content/eval_my_voice.py
"""Score the trained model on recordings it never saw during training.

Usage: python eval_my_voice.py --model my_custom_model/hey_millie.onnx --clips my_voice_holdout [--threshold 0.5]
"""
import argparse
import os
import sys
from pathlib import Path

# /content holds the cloned "openwakeword" repo folder, which would shadow the installed package
_here = os.path.dirname(os.path.abspath(__file__))
sys.path = [p for p in sys.path if os.path.abspath(p or ".") not in (_here, os.getcwd())]

import numpy as np
import scipy.io.wavfile as wavfile
from openwakeword.model import Model

p = argparse.ArgumentParser()
p.add_argument("--model", required=True)
p.add_argument("--clips", required=True)
p.add_argument("--threshold", type=float, default=0.5)
a = p.parse_args()

m = Model(wakeword_models=[a.model], inference_framework="onnx")
name = list(m.models.keys())[0]
files = sorted(Path(a.clips).glob("*.wav"))
hits = 0
for f in files:
    sr, x = wavfile.read(f)
    # add 1 s of silence on each side so the model has full context, like a live stream
    x = np.concatenate([np.zeros(16000, np.int16), x, np.zeros(16000, np.int16)])
    m.reset()
    scores = [m.predict(x[i:i + 1280])[name] for i in range(0, len(x) - 1280, 1280)]
    best = max(scores)
    hits += best >= a.threshold
    print(f"{f.name}: max score {best:.3f} {'DETECTED' if best >= a.threshold else 'missed'}")
print(f"\nDetected {hits}/{len(files)} held-out clips at threshold {a.threshold}")


Writing /content/eval_my_voice.py


In [12]:
# Convert every recording in your zip to 16 kHz mono WAV (the model's input format), then test
import zipfile, subprocess, shutil
from pathlib import Path
shutil.rmtree("/tmp/myvoice", ignore_errors=True); shutil.rmtree("my_voice_test", ignore_errors=True)
Path("my_voice_test").mkdir()
zipfile.ZipFile("/content/millie_voice_samples.zip").extractall("/tmp/myvoice")
for f in Path("/tmp/myvoice").rglob("*"):
    if f.suffix.lower() in {".wav", ".m4a", ".mp3"} and "__MACOSX" not in f.parts:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(f), "-ac", "1", "-ar", "16000",
                        "-sample_fmt", "s16", f"my_voice_test/{f.stem}.wav"], check=True)
print("test clips:", len(list(Path("my_voice_test").glob("*.wav"))))
!{PY} -W ignore eval_my_voice.py --model my_custom_model/hey_millie.onnx --clips my_voice_test --threshold 0.5

test clips: 6
hey_millie_01.wav: max score 0.689 DETECTED
hey_millie_02.wav: max score 0.614 DETECTED
hey_millie_03.wav: max score 0.274 missed
hey_millie_04.wav: max score 0.187 missed
hey_millie_05.wav: max score 0.389 missed
hey_millie_06.wav: max score 0.697 DETECTED

Detected 3/6 held-out clips at threshold 0.5


In [13]:
from google.colab import files
files.download("my_custom_model/hey_millie.onnx")
files.download("my_custom_model/hey_millie.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>